In [34]:
""" Get QoS """
import common_utils
import os
import pandas as pd
import difflib

# Example usage
root_folder = '../../../data_warehouse/minimized_warehouse_6bbb'
filename = 'worker1.feather'
subfolders = common_utils.find_subfolders_with_file(root_folder, filename)
print(subfolders)
prom_data_paths = {os.path.basename(x): x for x in subfolders}
yolo_data_paths = {key: os.path.join(val, "worker_qos.feather") for key, val in prom_data_paths.items()}


[]


In [31]:
"""
Get item count for each model and resolution
"""
num_items_dict = {}
for key in prom_data_paths.keys():
    try:
        yolo_df = common_utils.read_feather_cached(yolo_data_paths[key])
    except:
        print(f"Failed to read {key}")
        continue
    total_items = yolo_df["id"].max()
    model_info = common_utils.path_to_workers_and_pcl_size(key)
    if model_info.resolution not in num_items_dict:
        num_items_dict[model_info.resolution] = {}
    num_items_dict[model_info.resolution][model_info.num_vehicles] = total_items

In [27]:
num_items_dict.keys()
num_items_dict[10000].keys()
num_items_dict[10000][2]

25863

In [32]:
# Clean dataframe and calculate power
def get_total_joules(dataframe):
    cleaned_df = dataframe
    
    """ Sort by timestamp to make sure it makes sense to compute difference between first and last values """
    cleaned_df.sort_values(by="timestamp", inplace=True)
    
    """ Get all relevant columns for power calculation """
    target_word = 'kepler node package joules total dynamic'
    closest_matches = difflib.get_close_matches(target_word, cleaned_df.columns, n=2, cutoff=0.05)
    
    """ Compute joules per match """
    joules_per_match = []
    for match in closest_matches:
        joules = cleaned_df[match].max() - cleaned_df[match].min()
        joules_per_match.append(joules)
    
    """ Compute total joules """
    total_joules = sum(joules_per_match)
    return total_joules

total_joules_per_model = {}
for key in prom_data_paths.keys():
    paths = []
    model_info = common_utils.path_to_workers_and_pcl_size(key)

    """ Get all workers """
    for work_num in range(1, 6):
        temp_path = os.path.join(prom_data_paths[key], f"worker{work_num}.feather")
        paths.append(temp_path)

    """ Get joules per image for each worker """
    joules_per_worker = [get_total_joules(common_utils.get_cleaned_df(x)) for x in paths]
    joules_total = sum(joules_per_worker)
    num_images = num_items_dict[model_info.resolution][model_info.num_vehicles]
    joules_per_image = joules_total / num_images

    """ Add result to dict for current model and resolution """

    if model_info.resolution not in total_joules_per_model:
        total_joules_per_model[model_info.resolution] = {}
    total_joules_per_model[model_info.resolution][model_info.num_vehicles] = joules_per_image

max_joules = {}
for resolution in sorted(total_joules_per_model.keys()):
    joules = pd.DataFrame.from_dict(total_joules_per_model[resolution], orient='index', columns=['Joules'])
    joules.columns = [f'{resolution}']
    max_joules[resolution] = joules



In [33]:
# Grouped bars
import plotly.express as px
import numpy as np

# Define width based on resolution
# resolution_to_width = {160: 0.2, 320: 0.4, 640: 0.6, 1280: 0.8}
max_joules_df = pd.concat(max_joules.values(), axis=1)
max_joules_df_sorted = max_joules_df.sort_index()  # Sort by index first

# Create separator rows with NaN values
separator_row1 = pd.DataFrame(index=["..."], columns=max_joules_df_sorted.columns, data=np.nan)
separator_row2 = pd.DataFrame(index=["...."], columns=max_joules_df_sorted.columns,
                              data=np.nan)  # Using .... to make it unique

# Split the dataframe into three parts and insert the separators
mask1 = max_joules_df_sorted.index <= 10
mask2 = (max_joules_df_sorted.index > 10) & (max_joules_df_sorted.index <= 20)
mask3 = max_joules_df_sorted.index > 20

df_part1 = max_joules_df_sorted[mask1]
df_part2 = max_joules_df_sorted[mask2]
df_part3 = max_joules_df_sorted[mask3]

# Combine all parts with the separators
max_joules_df_sorted = pd.concat([df_part1, separator_row1, df_part2, separator_row2, df_part3])

# Convert remaining numeric indices to strings
max_joules_df_sorted.index = max_joules_df_sorted.index.astype(str)

fig = px.bar(max_joules_df_sorted, barmode='group', title='Joules per PCL',
             labels={'value': 'Max Power (Watts)', 'index': 'Model'})
fig.update_layout(xaxis_title='Num_workers', yaxis_title='Joules', legend_title_text='Resolution',
                  xaxis={'categoryorder': 'array', 'categoryarray': max_joules_df_sorted.index})
fig.show()

fig = px.bar(max_joules_df_sorted, barmode='group', title='Joules per PCL (Log Scale)',
             labels={'value': 'Max Power (Watts)', 'index': 'Model'})
fig.update_layout(xaxis_title='Num_workers', yaxis_title='Joules',
                  yaxis_type='log', legend_title_text='Resolution',
                  xaxis={'categoryorder': 'array', 'categoryarray': max_joules_df_sorted.index})
fig.show()


ValueError: No objects to concatenate